# 智能语音机器学习

**教学说明（本科生）：**

本节介绍语音信号处理和机器学习在语音理解中的应用。

**学习内容：**
1. 语音信号处理：波形、频谱、梅尔频谱、MFCC特征
2. 情绪识别：基于语音特征的情绪分类
3. 方言识别：普通话语音 vs 方言语音
4. 语音生成：Hugging Face的语音生成模型

**语音信号处理基础：**
- **波形**：语音的时域表示
- **频谱**：语音的频域表示（通过FFT）
- **梅尔频谱**：基于人耳听觉特性的频谱表示
- **MFCC**：梅尔频率倒谱系数，语音识别中最常用的特征

**核心概念：**

#### 四种语音表示对比

| 表示方式 | 维度 | 横轴 | 纵轴 | 优点 | 缺点 | 教学类比 |
|---------|------|------|------|------|------|---------|
| 波形 | 1D | 时间 | 振幅 | 最原始，信息完整 | 维度高，难直接使用 | 心电图 |
| 频谱 | 2D | 时间 | 频率(Hz) | 显示频率分布 | 维度高 | 钢琴琴键分布 |
| Mel频谱 | 2D | 时间 | Mel频率 | 符合人耳听觉 | 有信息损失 | 放大镜和望远镜 |
| MFCC | 2D | 时间 | 系数编号 | 信息紧凑，ML友好🔥 | 抽象难解释 | 身份证号码 |- **声学特征**：从语音波形中提取的可计算特征
- **分类模型**：使用SVM等传统机器学习方法进行语音分类
- **深度学习**：端到端的语音处理方法

**学习目标：**
1. 理解语音信号的基本特性
2. 掌握语音特征提取的方法
3. 体会机器学习在语音理解中的应用

### 普通话 vs 方言 的特征差异

| 特征 | 普通话 | 方言 |
|------|--------|------|
| MFCC系数 | 标准发音模式 | 可能偏移（口音） |
| 谱质心 | 较稳定 | 可能变化（发音习惯） |
| 过零率 | 标准范围 | 可能不同（语速、语调） |

> 💡 **教学提示**：方言识别和语音识别的区别在于——语音识别要听清"说了什么词"，方言识别要听出"是哪里的口音"。

---
### 学习重点

通过本节学习，你应该掌握：
- 理解上述概念的原理和意义
- 将理论知识与代码实现对应起来的能力


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sounddevice as sd
from scipy.io.wavfile import write

import librosa
import librosa.display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import whisper
import requests

In [ ]:
print("默认设备：")
print(sd.default.device)

print("\n可用音频设备：")
print(sd.query_devices())

## 观测自己的声音

**教学说明（本科生）：**

本节学习如何录制和查看自己的声音信号。

**学习内容：**
1. 使用sounddevice录制语音
2. 使用scipy保存音频文件
3. 使用librosa读取和显示音频

**核心概念：**
- **采样率**：每秒采集的样本数（常用16kHz）
- **波形**：语音信号的时域表示
- **音频文件格式**：WAV是最常用的无损格式

**学习目标：**
掌握语音数据采集和基本可视化方法。

### 语音信号处理"流水线"

```
语音波形(1D时域) → 短时傅里叶变换(STFT) → 频谱(2D时频) → Mel滤波器 → Mel频谱 → DCT → MFCC
```

> 💡 **教学提示**：从波形到MFCC，语音信号一步步被"精炼"——去掉不重要的信息，保留与语音内容最相关的特征。这就像从原材料（铁矿石）加工成可直接使用的产品（精铁）。

---
### 学习重点

通过本节学习，你应该掌握：
- 理解上述概念的原理和意义
- 如何通过可视化理解模型的内部行为
- 将理论知识与代码实现对应起来的能力


### 录制一段自己的语音

In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write

fs = 16000
duration = 3

print("开始录音...")

audio = sd.rec(
    int(duration * fs),
    samplerate=fs,
    channels=1
)

sd.wait()

write("student_voice.wav", fs, audio)

print("录音完成")

### 播放

In [ ]:
from IPython.display import Audio

Audio("student_voice.wav")

### 读取语音文件

In [ ]:
y, sr = librosa.load("student_voice.wav", sr=None)

print("语音数据读取完成")
print("采样率 sr =", sr)
print("语音数据长度 =", len(y))
print("语音时长 =", len(y) / sr, "秒")

### 绘制语音波形图

In [ ]:
plt.figure(figsize=(10, 4))

librosa.display.waveshow(y, sr=sr)

plt.title("Voice Waveform")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")

plt.tight_layout()
plt.show()

### 绘制频谱图

### 如何解读频谱图

- **X轴**：时间（语音随时间变化）
- **Y轴**：频率（从低音到高音）
- **颜色深浅**：能量强弱（深色=能量强）
- **横条纹**：基频及其谐波（决定音高）

频谱图是通过**短时傅里叶变换(STFT)** 生成的：将语音切成短片段→对每段做FFT→拼接到一起。

> 💡 **教学提示**：频谱图（Spectrogram）是语音处理的"标准视图"——它同时展示了时间、频率和能量三个维度的信息。对比不同人的频谱图，可以看到声纹的差异。

### 学习重点

- 理解本节代码的原理和实现
- 能够将输出结果与理论知识对应起来

---

In [ ]:
# 短时傅里叶变换
D = librosa.stft(y)

# 幅值谱
D_abs = np.abs(D)

# 转换为分贝尺度
D_db = librosa.amplitude_to_db(D_abs, ref=np.max)

plt.figure(figsize=(10, 4))

librosa.display.specshow(
    D_db,
    sr=sr,
    x_axis="time",
    y_axis="hz"
)

plt.colorbar(label="dB")
plt.title("Spectrogram")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")

plt.tight_layout()
plt.show()

### 绘制 Mel 频谱图

### Mel刻度的意义

人耳对低频更敏感（能区分200Hz和300Hz），对高频不敏感（很难区分2000Hz和2100Hz）。

Mel刻度模拟了这种特性：`mel = 2595 × log₁₀(1 + f/700)`

- **低频**：Mel刻度≈线性（变化率大）→ 分辨率高
- **高频**：Mel刻度≈对数（变化率小）→ 分辨率低

> 💡 **教学提示**：Mel频谱与普通频谱的区别在于频率刻度——Mel刻度仿照人耳特性，在低频区分辨率高，高频区分辨率低。这就像在低频区用"放大镜"看，高频区用"望远镜"看。

### 学习重点

- 理解本节代码的原理和实现
- 能够将输出结果与理论知识对应起来

---

In [ ]:
mel_spec = librosa.feature.melspectrogram(
    y=y,
    sr=sr,
    n_mels=128
)

mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

plt.figure(figsize=(10, 4))

librosa.display.specshow(
    mel_spec_db,
    sr=sr,
    x_axis="time",
    y_axis="mel"
)

plt.colorbar(label="dB")
plt.title("Mel Spectrogram")
plt.xlabel("Time (s)")
plt.ylabel("Mel Frequency")

plt.tight_layout()
plt.show()

### 提取 MFCC 特征

### 为什么取13个MFCC系数？

| 系数编号 | 大致代表 |
|---------|---------|
| MFCC 1 | 语音的整体能量 |
| MFCC 2~4 | 语音的频谱包络形状 |
| MFCC 5~8 | 语音的精细频谱细节 |
| MFCC 9~13 | 更高频的细节（通常已经较弱） |

13个系数是经验和理论的折中——太多了包含噪声，太少了丢失信息。

> 💡 **教学提示**：MFCC是语音识别中最常用的特征，它提取了语音的"频谱包络"信息，去掉了对语音识别不重要的细节。可以理解为——MFCC抓住了语音的"大趋势"而非"小波动"。

### 学习重点

- 理解本节代码的原理和实现
- 能够将输出结果与理论知识对应起来

---

In [ ]:
mfcc = librosa.feature.mfcc(
    y=y,
    sr=sr,
    n_mfcc=13
)

print("MFCC 特征矩阵形状：", mfcc.shape)

### 绘制 MFCC 特征图

> 💡 **教学提示**：MFCC图中横轴为时间，纵轴为13个系数。颜色深浅表示系数值大小。不同语音内容的MFCC图有明显差异，这使模型能够区分不同的语音。

### 学习重点

- 理解本节代码的原理和实现
- 能够将输出结果与理论知识对应起来

---

In [ ]:
plt.figure(figsize=(10, 4))

librosa.display.specshow(
    mfcc,
    sr=sr,
    x_axis="time"
)

plt.colorbar(label="MFCC value")
plt.title("MFCC Features")
plt.xlabel("Time (s)")
plt.ylabel("MFCC Coefficients")

plt.tight_layout()
plt.show()

## 情绪判断

**教学说明（本科生）：**

本节介绍基于语音的情绪识别任务。

**情绪分类：**
- normal（正常）
- happy（高兴）
- angry（生气）

**语音情绪特征：**
- **MFCC**：反映语音的频谱特性
- **过零率**：反映声音变化快慢
- **谱质心**：反映声音频率重心
- **RMS能量**：反映声音强弱

**学习目标：**
1. 理解语音情绪识别的基本思路
2. 掌握语音特征提取方法
3. 体会不同情绪在语音特征上的差异

### 语音情绪特征

| 特征 | 物理含义 | 与情绪的关系 |
|------|---------|-------------|
| MFCC | 频谱包络（反映声道形状） | 不同情绪改变发音方式 |
| 过零率 | 信号穿过零轴的频率 | 生气时→高，平静时→低 |
| 谱质心 | 频率的重心位置 | 高兴时→偏高，悲伤时→偏低 |
| RMS能量 | 声音的强度 | 生气时→高，平静时→低 |

> 💡 **教学提示**：情绪识别不仅仅看"说了什么"，更重要的是"怎么说"——语速、音调、音量等副语言信息。

---
### 学习重点

通过本节学习，你应该掌握：
- 理解上述概念的原理和意义
- 将理论知识与代码实现对应起来的能力


### 录制语音

> 💡 **教学提示**：请录制带有明显情绪的声音——例如生气时用力说话，开心时提高音调。情绪越明显，模型的分类效果越好。

### 学习重点

- 掌握语音数据采集的方法（采样率、时长、格式）
- 了解真实数据采集的挑战（环境噪声、发音一致性）

---

In [ ]:
data_dir = "voice_emotion_dataset"
os.makedirs(data_dir, exist_ok=True)

fs = 16000
duration = 3

label_names = ["normal", "happy", "angry"]
num_samples = 8

sentence = "今天的实验完成了"

for label_name in label_names:
    label_dir = os.path.join(data_dir, label_name)
    os.makedirs(label_dir, exist_ok=True)

    print(f"\n===== 开始录制情绪类别：{label_name} =====")
    print(f"请始终说同一句话：{sentence}")

    for i in range(num_samples):
        input(f"准备录制 {label_name} 的第 {i+1} 个样本，按回车开始...")

        print("开始录音...")

        audio = sd.rec(
            int(duration * fs),
            samplerate=fs,
            channels=1
        )

        sd.wait()

        filename = os.path.join(label_dir, f"{label_name}_{i+1}.wav")
        write(filename, fs, audio)

        print(f"已保存：{filename}")

print("\n所有情绪类别录制完成")

### 查看数据集结构

In [ ]:
data_dir = "voice_emotion_dataset"

for label in os.listdir(data_dir):
    label_path = os.path.join(data_dir, label)

    if os.path.isdir(label_path):
        wav_files = [f for f in os.listdir(label_path) if f.endswith(".wav")]
        print(label, ":", len(wav_files), "个wav文件")

### 随机读取一个语音样本

In [ ]:
sample_file = os.path.join(data_dir, "normal", "normal_7.wav")

y, sr = librosa.load(sample_file, sr=16000)

print("采样率：", sr)
print("语音长度：", len(y))
print("语音时长：", len(y) / sr, "秒")

### 绘制语音波形图

In [ ]:
plt.figure(figsize=(10, 4))

librosa.display.waveshow(y, sr=sr)

plt.title("Voice Waveform")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")

plt.tight_layout()
plt.show()

### 绘制频谱图

### 如何解读频谱图

- **X轴**：时间（语音随时间变化）
- **Y轴**：频率（从低音到高音）
- **颜色深浅**：能量强弱（深色=能量强）
- **横条纹**：基频及其谐波（决定音高）

频谱图是通过**短时傅里叶变换(STFT)** 生成的：将语音切成短片段→对每段做FFT→拼接到一起。

> 💡 **教学提示**：频谱图（Spectrogram）是语音处理的"标准视图"——它同时展示了时间、频率和能量三个维度的信息。对比不同人的频谱图，可以看到声纹的差异。

### 学习重点

- 理解本节代码的原理和实现
- 能够将输出结果与理论知识对应起来

---

In [ ]:
D = librosa.stft(y)
D_abs = np.abs(D)
D_db = librosa.amplitude_to_db(D_abs, ref=np.max)

plt.figure(figsize=(10, 4))

librosa.display.specshow(
    D_db,
    sr=sr,
    x_axis="time",
    y_axis="hz"
)

plt.colorbar(label="dB")
plt.title("Spectrogram")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")

plt.tight_layout()
plt.show()

### 绘制 Mel 频谱图

### Mel刻度的意义

人耳对低频更敏感（能区分200Hz和300Hz），对高频不敏感（很难区分2000Hz和2100Hz）。

Mel刻度模拟了这种特性：`mel = 2595 × log₁₀(1 + f/700)`

- **低频**：Mel刻度≈线性（变化率大）→ 分辨率高
- **高频**：Mel刻度≈对数（变化率小）→ 分辨率低

> 💡 **教学提示**：Mel频谱与普通频谱的区别在于频率刻度——Mel刻度仿照人耳特性，在低频区分辨率高，高频区分辨率低。这就像在低频区用"放大镜"看，高频区用"望远镜"看。

### 学习重点

- 理解本节代码的原理和实现
- 能够将输出结果与理论知识对应起来

---

In [ ]:
mel_spec = librosa.feature.melspectrogram(
    y=y,
    sr=sr,
    n_mels=128
)

mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

plt.figure(figsize=(10, 4))

librosa.display.specshow(
    mel_spec_db,
    sr=sr,
    x_axis="time",
    y_axis="mel"
)

plt.colorbar(label="dB")
plt.title("Mel Spectrogram")
plt.xlabel("Time (s)")
plt.ylabel("Mel Frequency")

plt.tight_layout()
plt.show()

### 绘制 MFCC 特征图

> 💡 **教学提示**：MFCC图中横轴为时间，纵轴为13个系数。颜色深浅表示系数值大小。不同语音内容的MFCC图有明显差异，这使模型能够区分不同的语音。

### 学习重点

- 理解本节代码的原理和实现
- 能够将输出结果与理论知识对应起来

---

In [ ]:
mfcc = librosa.feature.mfcc(
    y=y,
    sr=sr,
    n_mfcc=13
)

plt.figure(figsize=(10, 4))

librosa.display.specshow(
    mfcc,
    sr=sr,
    x_axis="time"
)

plt.colorbar(label="MFCC value")
plt.title("MFCC Features")
plt.xlabel("Time (s)")
plt.ylabel("MFCC Coefficients")

plt.tight_layout()
plt.show()

### 定义语音特征提取函数

In [ ]:
def extract_voice_features(file_path, sr=16000):
    """
    提取适合情绪识别的语音特征。
    
    特征包括：
    1. MFCC 均值和标准差
    2. 过零率均值和标准差
    3. 谱质心均值和标准差
    4. RMS 能量均值和标准差
    """

    y, sr = librosa.load(file_path, sr=sr)

    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std = np.std(mfcc, axis=1)

    # 过零率：反映声音变化快慢
    zcr = librosa.feature.zero_crossing_rate(y)
    zcr_mean = np.mean(zcr)
    zcr_std = np.std(zcr)

    # 谱质心：反映声音频率重心
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    centroid_mean = np.mean(spectral_centroid)
    centroid_std = np.std(spectral_centroid)

    # RMS能量：反映声音强弱
    rms = librosa.feature.rms(y=y)
    rms_mean = np.mean(rms)
    rms_std = np.std(rms)

    feature_vector = np.concatenate([
        mfcc_mean,
        mfcc_std,
        [zcr_mean, zcr_std],
        [centroid_mean, centroid_std],
        [rms_mean, rms_std]
    ])

    return feature_vector

### 构建特征矩阵

### 学习重点

- 理解"特征提取"在机器学习中的核心地位
- 原始语音波形 → 特征向量 的转换过程
- 特征设计直接影响模型性能

In [ ]:
X = []
y_labels = []
file_paths = []

for label in os.listdir(data_dir):
    label_path = os.path.join(data_dir, label)

    if not os.path.isdir(label_path):
        continue

    for file in os.listdir(label_path):
        if file.endswith(".wav"):
            file_path = os.path.join(label_path, file)

            feature = extract_voice_features(file_path)

            X.append(feature)
            y_labels.append(label)
            file_paths.append(file_path)

X = np.array(X)
y_labels = np.array(y_labels)

print("特征矩阵 X 的形状：", X.shape)
print("标签 y 的形状：", y_labels.shape)
print("类别：", np.unique(y_labels))

### 查看特征表

In [ ]:
feature_names = []

for i in range(13):
    feature_names.append(f"MFCC_{i+1}_mean")

for i in range(13):
    feature_names.append(f"MFCC_{i+1}_std")

feature_names += [
    "ZCR_mean",
    "ZCR_std",
    "SpectralCentroid_mean",
    "SpectralCentroid_std",
    "RMS_mean",
    "RMS_std"
]

df_features = pd.DataFrame(X, columns=feature_names)
df_features["label"] = y_labels
df_features["file"] = file_paths

df_features.head()

### 划分训练集和测试集

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_labels,
    test_size=0.3,
    random_state=42,
    stratify=y_labels
)

print("训练集数量：", X_train.shape[0])
print("测试集数量：", X_test.shape[0])
print("训练集类别：", np.unique(y_train))
print("测试集类别：", np.unique(y_test))

### 训练情绪识别模型

### SVM 的直观理解

支持向量机(SVM)的核心思想是：在特征空间中找到一个"最佳分界线"，使得不同类别的数据点离这条线尽可能远。

就像用筷子分开盘中混合的红豆和绿豆——筷子（决策边界）放得越巧妙，分开的效果越好。

这里使用的**RBF核**可以处理非线性边界，适合语音这种高维、复杂的特征空间。

### 学习重点

- 理解本节代码的原理和实现
- 能够将输出结果与理论知识对应起来

---

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", probability=True, random_state=42))
])

model.fit(X_train, y_train)

print("模型训练完成")

### 模型预测与评价

In [ ]:
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)

print("模型准确率：", acc)
print()
print("分类报告：")
print(classification_report(y_test, y_pred))

### 混淆矩阵可视化

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
labels = model.classes_

plt.figure(figsize=(5, 4))

plt.imshow(cm)

plt.xticks(range(len(labels)), labels)
plt.yticks(range(len(labels)), labels)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")

for i in range(len(labels)):
    for j in range(len(labels)):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.tight_layout()
plt.show()

### 录制一段新语音并分类情绪

In [ ]:
fs = 16000
duration = 3
test_file = "new_emotion_test.wav"

print("请仍然说同一句话：今天的实验完成了")
print("开始录制待预测语音...")

audio = sd.rec(
    int(duration * fs),
    samplerate=fs,
    channels=1
)

sd.wait()

write(test_file, fs, audio)

print("录制完成")

### 开始分类

In [ ]:
new_feature = extract_voice_features(test_file)
new_feature = new_feature.reshape(1, -1)

pred_label = model.predict(new_feature)[0]
pred_prob = model.predict_proba(new_feature)[0]

print("预测情绪类别：", pred_label)
print("\n各类别概率：")

for label, prob in zip(model.classes_, pred_prob):
    print(label, ":", round(prob, 4))

## 普通话 vs 方言

**教学说明（本科生）：**

本节介绍普通话语音和方言语音的识别任务。

**任务目标：**
- 分类不同语言变体
- 理解语音中的语言特征

**学习内容：**
1. 语音数据录制
2. 特征提取和模型训练
3. 模型性能评估

**学习目标：**
理解语音识别中的语言变体问题，体会深度学习在跨语言任务中的应用。

### 普通话 vs 方言 的特征差异

| 特征 | 普通话 | 方言 |
|------|--------|------|
| MFCC系数 | 标准发音模式 | 可能偏移（口音） |
| 谱质心 | 较稳定 | 可能变化（发音习惯） |
| 过零率 | 标准范围 | 可能不同（语速、语调） |

> 💡 **教学提示**：方言识别和语音识别的区别在于——语音识别要听清"说了什么词"，方言识别要听出"是哪里的口音"。

---
### 学习重点

通过本节学习，你应该掌握：
- 理解上述概念的原理和意义
- 模型训练和评估的基本流程
- 将理论知识与代码实现对应起来的能力


### 录制语音

> 💡 **教学提示**：请录制带有明显情绪的声音——例如生气时用力说话，开心时提高音调。情绪越明显，模型的分类效果越好。

### 学习重点

- 掌握语音数据采集的方法（采样率、时长、格式）
- 了解真实数据采集的挑战（环境噪声、发音一致性）

---

In [ ]:
import os
import sounddevice as sd
from scipy.io.wavfile import write

data_dir = "dialect_voice_dataset"
os.makedirs(data_dir, exist_ok=True)

fs = 16000
duration = 2
num_samples = 3

label_info = {
    "mandarin_hello": "普通话：你好",
    "mandarin_goodbye": "普通话：再见",
    "dialect_hello": "方言：你好",
    "dialect_goodbye": "方言：再见"
}

for label_name, description in label_info.items():
    label_dir = os.path.join(data_dir, label_name)
    os.makedirs(label_dir, exist_ok=True)

    print(f"\n===== 开始录制类别：{label_name} =====")
    print(f"请录制内容：{description}")

    for i in range(num_samples):
        input(f"准备录制 {description} 的第 {i+1} 个样本，按回车开始...")

        audio = sd.rec(
            int(duration * fs),
            samplerate=fs,
            channels=1
        )

        sd.wait()

        filename = os.path.join(label_dir, f"{label_name}_{i+1}.wav")
        write(filename, fs, audio)

        print(f"已保存：{filename}")

print("\n所有方言语音数据录制完成")

### 翻译字典

In [ ]:
translation_dict = {
    "mandarin_hello": "你好",
    "mandarin_goodbye": "再见",
    "dialect_hello": "你好",
    "dialect_goodbye": "再见"
}

### 构建特征矩阵

### 学习重点

- 理解"特征提取"在机器学习中的核心地位
- 原始语音波形 → 特征向量 的转换过程
- 特征设计直接影响模型性能

In [ ]:
def extract_mfcc_feature(file_path, n_mfcc=13):

    y, sr = librosa.load(file_path, sr=16000)

    mfcc = librosa.feature.mfcc(

        y=y,

        sr=sr,

        n_mfcc=n_mfcc

    )

    mfcc_mean = np.mean(mfcc, axis=1)

    mfcc_std = np.std(mfcc, axis=1)

    feature = np.concatenate([mfcc_mean, mfcc_std])

    return feature

X = []

y_labels = []

for label in os.listdir(data_dir):

    label_path = os.path.join(data_dir, label)

    if not os.path.isdir(label_path):

        continue

    for file in os.listdir(label_path):

        if file.endswith(".wav"):

            file_path = os.path.join(label_path, file)

            feature = extract_mfcc_feature(file_path)

            X.append(feature)

            y_labels.append(label)

X = np.array(X)

y_labels = np.array(y_labels)

print("特征矩阵形状：", X.shape)

print("标签数量：", y_labels.shape)

print("类别：", np.unique(y_labels))

### 模型训练

> 💡 **教学提示**：SVM在小样本分类任务上表现优异。这里使用RBF核来处理语音特征的非线性关系。

### 学习重点

- 理解本节代码的原理和实现
- 能够将输出结果与理论知识对应起来

---

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_labels,
    test_size=0.2,
    random_state=42,
    stratify=y_labels
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", probability=True, random_state=42))
])

model.fit(X_train, y_train)

print("方言语音分类模型训练完成")

### 模型评价

In [ ]:
y_pred = model.predict(X_test)

print("准确率：", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

### 模型测试

In [ ]:
test_file = "new_dialect_test.wav"

print("请说一句：你好 或 再见，可以用普通话或方言")
input("按回车开始录音...")

audio = sd.rec(
    int(duration * fs),
    samplerate=fs,
    channels=1
)

sd.wait()

write(test_file, fs, audio)

new_feature = extract_mfcc_feature(test_file)
new_feature = new_feature.reshape(1, -1)

pred_label = model.predict(new_feature)[0]
pred_prob = model.predict_proba(new_feature)[0]

print("模型识别类别：", pred_label)
print("普通话翻译结果：", translation_dict[pred_label])

print("\n各类别概率：")
for label, prob in zip(model.classes_, pred_prob):
    print(label, ":", round(prob, 4))

# Hugging Face Space

**教学说明（本科生）：**

本节介绍Hugging Face上的语音生成模型。

**Hugging Face Space：**
- 在线演示机器学习模型的平台
- 支持各种AI应用演示

**推荐模型：**
1. **音乐生成**：SongGeneration - 根据输入生成音乐
2. **语音克隆**：OmniVoice - 语音复制和转换
3. **格式转换**：LTX2.3-Studio - 音频格式转换

**学习目标：**
1. 了解Hugging Face平台
2. 体验前沿语音生成技术
3. 思考AI语音技术的应用场景

### 完整的语音处理流水线知识回顾

```
音频录制 → 信号可视化 → 特征提取(MFCC等) → ML模型(SVM) → 预测分类
    ↓          ↓              ↓                  ↓            ↓
 sounddevice  波形/频谱      librosa           sklearn      confidence
```

**本实验覆盖的知识点**：
1. ✅ 语音信号时域分析（波形）
2. ✅ 语音信号频域分析（频谱、Mel频谱）
3. ✅ 特征提取（MFCC、过零率、谱质心、RMS）
4. ✅ 传统机器学习分类（SVM）
5. ✅ 情绪识别 & 方言识别 两个实际应用

---
### 学习重点

通过本节学习，你应该掌握：
- 理解上述概念的原理和意义
- 如何通过可视化理解模型的内部行为
- 将理论知识与代码实现对应起来的能力


## 音乐生成
- https://huggingface.co/spaces/tencent/SongGeneration

## 语音克隆 & 生成
- https://huggingface.co/spaces/k2-fsa/OmniVoice

## 格式转换
- https://huggingface.co/spaces/techfreakworm/LTX2.3-Studio

### 学习重点

- 理解本节代码的原理和实现
- 能够将输出结果与理论知识对应起来

---